In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
from scipy.integrate import solve_ivp
from scipy.signal import argrelextrema

plt.rcParams.update({
    "font.size": 10,
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

FIGDIR = "figs"
os.makedirs(FIGDIR, exist_ok=True)
SIGMA, BETA = 10.0, 8 / 3

print("Setup complete.")

Setup complete.


In [7]:
def logistic_timeseries(r, x0=0.5, n_iter=1000, transient=300):
    x = np.empty(n_iter)
    x[0] = x0
    for n in range(n_iter - 1):
        x[n + 1] = r * x[n] * (1 - x[n])
    return x

r_demo = 3.9
x_series = logistic_timeseries(r_demo, x0=0.5, n_iter=1000, transient=300)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(np.arange(len(x_series)), x_series, lw=0.8, color="#1f77b4")
ax.set_xlabel("iteration $n$")
ax.set_ylabel("$x_n$")
ax.set_title(f"Logistic map time series ($r={r_demo}$, $x_0=0.5$)")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/logistic_timeseries.png")
plt.close(fig)

r_vals = np.arange(2.5, 4.0, 0.002)
n_iter = 1000
transient = 500

x = np.full_like(r_vals, 0.5)
r_plot = []
x_plot = []
for n in range(n_iter):
    x = r_vals * x * (1 - x)
    if n >= transient:
        r_plot.append(r_vals)
        x_plot.append(x.copy())
r_plot = np.concatenate(r_plot)
x_plot = np.concatenate(x_plot)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(r_plot, x_plot, ",", color="black", alpha=0.35, markersize=0.1)
ax.set_xlabel("$r$")
ax.set_ylabel("$x_n$ (retained)")
ax.set_title("Logistic map bifurcation diagram")
ax.set_xlim(2.5, 4.0)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/logistic_bifurcation.png")
plt.close(fig)

print("Logistic map done.")

Logistic map done.


In [8]:
def henon_iterate(a, b, x0=0.0, y0=0.0, n_iter=5000, transient=500):
    x = np.empty(n_iter)
    y = np.empty(n_iter)
    x[0], y[0] = x0, y0
    for n in range(n_iter - 1):
        x[n + 1] = 1 - a * x[n] ** 2 + y[n]
        y[n + 1] = b * x[n]
    return x[transient:], y[transient:]

a_c, b_c = 1.4, 0.3
hx, hy = henon_iterate(a_c, b_c, n_iter=5000, transient=500)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(hx, hy, s=0.3, color="#d62728", alpha=0.6)
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title(f"Henon attractor ($a={a_c}$, $b={b_c}$)")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/henon_attractor.png")
plt.close(fig)

a_vals = np.arange(1.0, 1.4, 0.002)
b_fixed = 0.3
n_iter = 1000
transient = 500

x = np.zeros_like(a_vals)
y = np.zeros_like(a_vals)
a_plot = []
x_plot2 = []
for n in range(n_iter):
    x_new = 1 - a_vals * x ** 2 + y
    y_new = b_fixed * x
    x, y = x_new, y_new
    if n >= transient:
        a_plot.append(a_vals)
        x_plot2.append(x.copy())
a_plot = np.concatenate(a_plot)
x_plot2 = np.concatenate(x_plot2)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(a_plot, x_plot2, ",", color="black", alpha=0.35, markersize=0.1)
ax.set_xlabel("$a$")
ax.set_ylabel("$x_n$ (retained)")
ax.set_title(f"Henon map bifurcation diagram ($b={b_fixed}$ fixed)")
ax.set_xlim(1.0, 1.4)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/henon_bifurcation.png")
plt.close(fig)

print("Henon map done.")

Henon map done.


In [9]:
def phase_and_timeseries(t, sol, name, labels=("x", "y", "z"), proj_pairs=None, color="#1f77b4"):
    """Save a 3D phase plot (if 3 states) or 2D projections (if >3 states) plus a time-series figure."""
    n_states = sol.shape[0]

    if n_states == 3:
        fig = plt.figure(figsize=(6, 5.5))
        ax = fig.add_subplot(111, projection="3d")
        ax.plot(sol[0], sol[1], sol[2], lw=0.4, color=color)
        ax.set_xlabel(labels[0])
        ax.set_ylabel(labels[1])
        ax.set_zlabel(labels[2])
        ax.set_title(f"{name}: phase-space trajectory")
        fig.tight_layout()
        fig.savefig(f"{FIGDIR}/{name.lower()}_phase3d.png")
        plt.close(fig)
    else:
        n_proj = len(proj_pairs)
        fig, axes = plt.subplots(1, n_proj, figsize=(4.2 * n_proj, 4))
        if n_proj == 1:
            axes = [axes]
        for ax, (i, j) in zip(axes, proj_pairs):
            ax.plot(sol[i], sol[j], lw=0.3, color=color)
            ax.set_xlabel(labels[i])
            ax.set_ylabel(labels[j])
            ax.set_title(f"{labels[i]}-{labels[j]} projection")
        fig.suptitle(f"{name}: phase-space projections")
        fig.tight_layout()
        fig.savefig(f"{FIGDIR}/{name.lower()}_phase_proj.png")
        plt.close(fig)

    fig, axes = plt.subplots(n_states, 1, figsize=(7, 1.6 * n_states), sharex=True)
    if n_states == 1:
        axes = [axes]
    for k in range(n_states):
        axes[k].plot(t, sol[k], lw=0.6, color=color)
        axes[k].set_ylabel(labels[k])
        axes[k].grid(alpha=0.3)
    axes[-1].set_xlabel("t")
    fig.suptitle(f"{name}: time series")
    fig.tight_layout()
    fig.savefig(f"{FIGDIR}/{name.lower()}_timeseries.png")
    plt.close(fig)

print("Helper functions ready.")

Helper functions ready.


In [10]:
def lorenz_rhs(state, rho, sigma=SIGMA, beta=BETA):
    x, y, z = state
    return np.array([sigma * (y - x), x * (rho - z) - y, x * y - beta * z])


def rk4_step(state, rho, dt):
    k1 = lorenz_rhs(state, rho)
    k2 = lorenz_rhs(state + 0.5 * dt * k1, rho)
    k3 = lorenz_rhs(state + 0.5 * dt * k2, rho)
    k4 = lorenz_rhs(state + dt * k3, rho)
    return state + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def rk4_integrate(state0, rho, dt, T):
    n_steps = int(round(T / dt))
    states = np.empty((n_steps + 1, 3))
    states[0] = state0
    s = np.array(state0, dtype=float)
    for i in range(n_steps):
        s = rk4_step(s, rho, dt)
        states[i + 1] = s
    t = np.arange(n_steps + 1) * dt
    return t, states


rho_vals = np.arange(0.0, 30.0 + 1e-9, 0.1)
dt = 0.01
T = 60.0
transient_t = 20.0
state0 = np.array([1.0, 1.0, 1.0])

rho_points = []
z_points = []

for rho in rho_vals:
    t, states = rk4_integrate(state0, rho, dt, T)
    mask = t >= transient_t
    z = states[mask, 2]
    idx = argrelextrema(z, np.greater)[0]
    if len(idx) == 0:
        z_points.append(z[-1])
        rho_points.append(rho)
    else:
        z_points.extend(z[idx])
        rho_points.extend([rho] * len(idx))

rho_points = np.array(rho_points)
z_points = np.array(z_points)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(rho_points, z_points, ",", color="black", alpha=0.4, markersize=0.3)
ax.set_xlabel(r"$\rho$")
ax.set_ylabel("local maxima of $z(t)$")
ax.set_title(r"Lorenz sensitivity to $\rho$ ($\sigma=10$, $\beta=8/3$)")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/lorenz_rho_sweep.png")
plt.close(fig)
print("rho-sweep done.")

rho-sweep done.


In [11]:
rho_std = 28.0
T2 = 50.0
state0_2 = np.array([1.0, 1.0, 1.0])
dts = [0.001, 0.01, 0.05]
labels = ["dt=0.001 (fine)", "dt=0.01 (baseline)", "dt=0.05 (coarse)"]
colors = ["#1f77b4", "#2ca02c", "#d62728"]

results = {}
for dt_i in dts:
    t_i, s_i = rk4_integrate(state0_2, rho_std, dt_i, T2)
    results[dt_i] = (t_i, s_i)

fig, ax = plt.subplots(figsize=(8, 3.5))
for dt_i, lab, col in zip(dts, labels, colors):
    t_i, s_i = results[dt_i]
    ax.plot(t_i, s_i[:, 0], lw=0.7, label=lab, color=col, alpha=0.85)
ax.set_xlabel("t")
ax.set_ylabel("x(t)")
ax.set_title("Step-size comparison: x(t) overlay")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/stepsize_x_overlay.png")
plt.close(fig)

fig = plt.figure(figsize=(6.5, 5.5))
ax = fig.add_subplot(111, projection="3d")
for dt_i, lab, col in zip(dts, labels, colors):
    t_i, s_i = results[dt_i]
    ax.plot(s_i[:, 0], s_i[:, 1], s_i[:, 2], lw=0.4, label=lab, color=col, alpha=0.7)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Step-size comparison: phase-space overlay")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/stepsize_phase_overlay.png")
plt.close(fig)

t_base, s_base = results[0.01]
fig, ax = plt.subplots(figsize=(8, 3.5))
for dt_i, lab, col in zip([0.001, 0.05], [labels[0], labels[2]], [colors[0], colors[2]]):
    t_i, s_i = results[dt_i]
    x_interp = np.interp(t_base, t_i, s_i[:, 0])
    diff = np.abs(x_interp - s_base[:, 0])
    ax.semilogy(t_base, diff + 1e-16, lw=0.8, label=f"|{lab.split(' ')[0]} - baseline|", color=col)
ax.set_xlabel("t")
ax.set_ylabel(r"$|x_{dt}(t)-x_{0.01}(t)|$ (log scale)")
ax.set_title("Pairwise divergence from baseline (dt=0.01)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/stepsize_divergence.png")
plt.close(fig)

print("Step-size sensitivity done.")
print("Part C complete.")

Step-size sensitivity done.
Part C complete.
